# Kontrata — Qwen2.5-1.5B LoRA fine-tune

Bu defter **Google Colab GPU** (T4 veya üzeri) içindir. Yerel Mac (8 GB RAM) eğitim için yeterli değildir.

Çalıştırmadan önce:
1. Runtime → Change runtime type → GPU
2. 🔑 simgesinden `HF_TOKEN` secret'ı ekleyin (Hugging Face yazma yetkisi). Jeton koda gömülmez.
3. `ml/data/train.jsonl` ve `val.jsonl` yoksa yerelde `python generate.py --seed 42` çalıştırıp hücre 2'de yükleyin; veya `VERI_KAYNAGI = "github"` ile depodan üretin.

Çıktı:
- Adapter: `fatmaoz/kontrata-qwen-lora-v1`
- Endpoint için birleşik model: `fatmaoz/kontrata-qwen-merged-v1`


In [ ]:
# Hücre 1 — kurulum
# Taban paketler. Jeton yalnızca Colab secrets'tan okunur.

%pip install -q transformers peft trl bitsandbytes accelerate datasets huggingface_hub jsonschema

import subprocess
import sys

print("=== GPU ===")
gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
if gpu.returncode != 0:
    raise RuntimeError("nvidia-smi başarısız. Runtime → Change runtime type → T4 GPU seçin.")

import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA yok. Colab'da GPU çalışma zamanı açın.")
print(f"cuda: {torch.cuda.get_device_name(0)}")

from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Colab Secrets'a HF_TOKEN ekleyin; koda yapıştırmayın.")
login(token=HF_TOKEN)
print("Hugging Face oturumu açıldı (jeton yazdırılmadı).")


In [ ]:
# Hücre 2 — veri
# Yol A: dosyalar /content altındaysa doğrudan kullan, yoksa files.upload
# Yol B: GitHub klonu + generate.py (jsonl sürüme girmez)

import json
import subprocess
import sys
import urllib.request
from pathlib import Path

from datasets import Dataset
from google.colab import files

# "upload" veya "github"
VERI_KAYNAGI = "github"

REPO_CLONE = Path("/content/kontrata")
SCHEMA_URL = "https://raw.githubusercontent.com/oz-fatma/kontrata/main/ml/schema/kontrat.json"
CONTENT = Path("/content")
TRAIN_PATH = CONTENT / "train.jsonl"
VAL_PATH = CONTENT / "val.jsonl"
SCHEMA_PATH = CONTENT / "kontrat.json"

SYSTEM_PROMPT = """Sen bir kontenjan sözleşmesi çıkarım motorusun. Verilen sözleşme metninden JSON üret.

SADECE JSON döndür. Tablo, markdown, açıklama YAZMA.

Çıktı tam olarak şu biçimde olmalı:
{"meta":{"otel_adi":"Argos Otel","acente_adi":"Side Turizm","para_birimi":"GBP","kur_esasi":"giris_gunu_tcmb","yetkili_mahkeme":"Antalya"},"donem":{"baslangic":"2026-04-01","bitis":"2026-10-31","alt_donemler":[]},"oda_kontenjanlari":[{"oda_tipi":"standart","adet":170}],"fiyatlar":[{"oda_tipi":"standart","tutar":50,"birim":"oda_gecelik","pansiyon":"belirtilmemis"}],"release":{"gun":10,"kapsam":"isim_listesi"},"stop_sale":[]}

Alan kuralları:
- meta (isteğe bağlı, sadece şu alanlar): otel_adi, acente_adi, para_birimi (EUR|GBP|USD|TRY), kur_esasi (giris_gunu_tcmb|cikis_gunu_tcmb|sabit_kur|belirtilmemis), yetkili_mahkeme (sadece şehir adı), sozlesme_tipi (tamamen_garantili|kismen_garantili|garantisiz|istege_bagli|serbest_satis|blok_rezervasyon|blok_satin_alma|belirtilmemis), sezon (yaz|kis|yillik|belirtilmemis)
- donem.baslangic, donem.bitis: ISO tarih veya null
- oda_kontenjanlari: oda_tipi (standart/suit/balayi/engelli/aile/deluxe), adet (tam sayı)
- fiyatlar: oda_tipi, tutar (sayı), birim (oda_gecelik|kisi_gecelik), pansiyon (RO|BB|HB|FB|AI|belirtilmemis)
- release: gun (tam sayı), kapsam (isim_listesi|kontenjan_iadesi|her_ikisi|belirtilmemis)
- stop_sale: dizi, yoksa []

Tek JSON nesnesi. Bittiğinde dur.
meta alanı yalnızca en üstte bir kez yazılır. Diğer alanların içine meta bilgisi (yetkili_mahkeme, para_birimi vb.) yazma.
"""


def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def ensure_schema() -> Path:
    if SCHEMA_PATH.exists():
        return SCHEMA_PATH
    cloned = REPO_CLONE / "ml" / "schema" / "kontrat.json"
    if cloned.exists():
        SCHEMA_PATH.write_text(cloned.read_text(encoding="utf-8"), encoding="utf-8")
        return SCHEMA_PATH
    urllib.request.urlretrieve(SCHEMA_URL, SCHEMA_PATH)
    return SCHEMA_PATH


if VERI_KAYNAGI == "upload":
    if TRAIN_PATH.exists() and VAL_PATH.exists():
        print("train.jsonl ve val.jsonl /content altında bulundu, yükleme atlandı.")
    else:
        print("train.jsonl ve val.jsonl seçin (ikisini birlikte seçin: Cmd ile çoklu seçim).")
        uploaded = files.upload()
        names = {Path(n).name: n for n in uploaded}
        if "train.jsonl" not in names or "val.jsonl" not in names:
            raise RuntimeError("Hem train.jsonl hem val.jsonl yüklenmeli.")
        if names["train.jsonl"] != "train.jsonl":
            Path(names["train.jsonl"]).replace(TRAIN_PATH)
        if names["val.jsonl"] != "val.jsonl":
            Path(names["val.jsonl"]).replace(VAL_PATH)
elif VERI_KAYNAGI == "github":
    if not REPO_CLONE.exists():
        subprocess.check_call(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/oz-fatma/kontrata.git",
                str(REPO_CLONE),
            ]
        )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faker"])
    subprocess.check_call(
        [sys.executable, str(REPO_CLONE / "ml" / "generate.py"), "--seed", "42"]
    )
    TRAIN_PATH = REPO_CLONE / "ml" / "data" / "train.jsonl"
    VAL_PATH = REPO_CLONE / "ml" / "data" / "val.jsonl"
    SCHEMA_PATH = REPO_CLONE / "ml" / "schema" / "kontrat.json"
else:
    raise RuntimeError('VERI_KAYNAGI "upload" veya "github" olmalı')

ensure_schema()
train_rows = load_jsonl(TRAIN_PATH)
val_rows = load_jsonl(VAL_PATH)
print(f"train={len(train_rows)} val={len(val_rows)}")


def to_messages(row: dict) -> dict:
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["metin"]},
            {
                "role": "assistant",
                "content": json.dumps(row["cikti"], ensure_ascii=False),
            },
        ]
    }


train_ds = Dataset.from_list(train_rows).map(
    to_messages, remove_columns=list(train_rows[0].keys())
)
val_ds = Dataset.from_list(val_rows).map(
    to_messages, remove_columns=list(val_rows[0].keys())
)
print(train_ds[0]["messages"][0]["role"], "sistem talimatı hazır")


In [ ]:
# Hücre 3 — model
# Qwen2.5-1.5B-Instruct, 4-bit nf4 + double quant, LoRA r=16

import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_DIR = "/content/kontrata-qwen-lora"
MERGED_DIR = "/content/kontrata-qwen-merged"

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

print(f"taban: {BASE_MODEL}")
print(f"dtype: {compute_dtype}, LoRA r=16 alpha=32 dropout=0.05")


In [ ]:
# Hücre 4 — eğitim
# SFTTrainer: 4 epoch, batch 4, grad acc 4, lr 2e-4, cosine, warmup 5 adım, max_len 1024
# Not: hız için gradient_checkpointing kapalı, max_length 1024
# Eğitim biter bitmez adapter HF'ye yüklenir (Colab düşerse kaybolmasın)

import time

from trl import SFTConfig, SFTTrainer

MAX_SEQ_LENGTH = 1024
HF_REPO_ID = "fatmaoz/kontrata-qwen-lora-v1"

sft_args = SFTConfig(
    output_dir=ADAPTER_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    max_length=MAX_SEQ_LENGTH,
    report_to="none",
    optim="paged_adamw_8bit",
    gradient_checkpointing=False,
    bf16=compute_dtype == torch.bfloat16,
    fp16=compute_dtype != torch.bfloat16,
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_config,
    processing_class=tokenizer,
)

t0 = time.perf_counter()
train_result = trainer.train()
elapsed = time.perf_counter() - t0
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

try:
    from huggingface_hub import HfApi

    api = HfApi()
    api.create_repo(HF_REPO_ID, exist_ok=True, private=True)
    api.upload_folder(folder_path=ADAPTER_DIR, repo_id=HF_REPO_ID)
    print(f"adapter HF'ye yuklendi: {HF_REPO_ID}")
except Exception as e:
    print(f"HF yukleme basarisiz: {type(e).__name__}: {e}")

history = trainer.state.log_history
train_pts = [(h.get("step"), h["loss"]) for h in history if "loss" in h]
eval_pts = [(h.get("epoch"), h["eval_loss"]) for h in history if "eval_loss" in h]

print(f"eğitim süresi: {elapsed / 60:.2f} dakika ({elapsed:.0f} sn)")
print(f"son train loss: {train_result.training_loss:.4f}")
print("epoch sonu val kaybı:")
for epoch, loss in eval_pts:
    print(f"  epoch={epoch:.2f}  eval_loss={loss:.4f}")

print("kayıp eğrisi (adım, train_loss):")
for step, loss in train_pts:
    print(f"  step={step}  loss={loss:.4f}")

try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    if train_pts:
        axes[0].plot([p[0] for p in train_pts], [p[1] for p in train_pts])
    axes[0].set_xlabel("adım")
    axes[0].set_ylabel("train loss")
    axes[0].set_title("eğitim kaybı")
    if eval_pts:
        axes[1].plot([p[0] for p in eval_pts], [p[1] for p in eval_pts], marker="o")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("eval_loss")
    axes[1].set_title("doğrulama kaybı")
    fig.tight_layout()
    plt.show()
except Exception as e:
    print(f"grafik çizilemedi: {type(e).__name__}")

In [ ]:
# Hücre 5 — hızlı değerlendirme
# val setinden 20 örnek; geçerli JSON, şema uyumu, alan doğruluğu
# extract_json ardışık JSON nesnelerini çözüp tek sözlükte birleştirir

import json
import re
from collections import OrderedDict
from pathlib import Path

from jsonschema import Draft202012Validator, FormatChecker

CORE_FIELDS = ("donem", "oda_kontenjanlari", "fiyatlar", "release", "stop_sale")
N_EVAL = 20
MAX_NEW_TOKENS = 1536


def extract_json(text: str):
    if not text or not str(text).strip():
        return None
    raw = str(text).strip()
    fence = re.search(r"```(?:json)?\s*([\s\S]*?)```", raw, re.IGNORECASE)
    if fence:
        raw = fence.group(1).strip()

    dec = json.JSONDecoder()
    birlesik, i, n = {}, 0, len(raw)
    while i < n:
        while i < n and raw[i] != "{":
            i += 1
        if i >= n:
            break
        try:
            obj, son = dec.raw_decode(raw, i)
        except json.JSONDecodeError:
            i += 1
            continue
        if isinstance(obj, dict):
            birlesik.update(obj)
        i = son
    return birlesik or None


def _sort_key(item: dict) -> str:
    return json.dumps(item, sort_keys=True, ensure_ascii=False)


def canonicalize_field(name: str, value):
    if value is None:
        return None
    if name == "donem" and isinstance(value, dict):
        alts = []
        for a in value.get("alt_donemler") or []:
            if isinstance(a, dict):
                alts.append(
                    {"ad": a.get("ad"), "baslangic": a.get("baslangic"), "bitis": a.get("bitis")}
                )
        alts.sort(key=_sort_key)
        return {"baslangic": value.get("baslangic"), "bitis": value.get("bitis"), "alt_donemler": alts}
    if name == "oda_kontenjanlari" and isinstance(value, list):
        rows = [
            {"oda_tipi": x.get("oda_tipi"), "adet": x.get("adet")}
            for x in value
            if isinstance(x, dict)
        ]
        rows.sort(key=_sort_key)
        return rows
    if name == "fiyatlar" and isinstance(value, list):
        rows = []
        for x in value:
            if not isinstance(x, dict):
                continue
            tutar = x.get("tutar")
            if isinstance(tutar, (int, float)):
                tutar = float(tutar)
            rows.append(
                {
                    "oda_tipi": x.get("oda_tipi"),
                    "tutar": tutar,
                    "birim": x.get("birim"),
                    "pansiyon": x.get("pansiyon") or "belirtilmemis",
                    "alt_donem_ad": x.get("alt_donem_ad"),
                }
            )
        rows.sort(key=_sort_key)
        return rows
    if name == "release" and isinstance(value, dict):
        return {"gun": value.get("gun"), "kapsam": value.get("kapsam") or "belirtilmemis"}
    if name == "stop_sale" and isinstance(value, list):
        rows = [
            {
                "baslangic": x.get("baslangic"),
                "bitis": x.get("bitis"),
                "kapsam": x.get("kapsam"),
                "bildirim_yontemi": x.get("bildirim_yontemi") or "belirtilmemis",
            }
            for x in value
            if isinstance(x, dict)
        ]
        rows.sort(key=_sort_key)
        return rows
    return value


def field_matches(gold, pred) -> dict:
    gold_obj = gold if isinstance(gold, dict) else {}
    pred_obj = pred if isinstance(pred, dict) else {}
    return {
        name: canonicalize_field(name, gold_obj.get(name))
        == canonicalize_field(name, pred_obj.get(name))
        for name in CORE_FIELDS
    }


schema = json.loads(Path(SCHEMA_PATH).read_text(encoding="utf-8"))
validator = Draft202012Validator(schema, format_checker=FormatChecker())

model.eval()
model.config.use_cache = True
device = next(model.parameters()).device

eos_ids = [tokenizer.eos_token_id]
im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
if isinstance(im_end, int) and im_end >= 0 and im_end not in eos_ids:
    eos_ids.append(im_end)

sample = val_rows[:N_EVAL]
records = []

for i, row in enumerate(sample):
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["metin"]},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.pad_token_id,
        )
    gen = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)
    parsed = extract_json(gen)
    valid_json = isinstance(parsed, dict)
    schema_ok = valid_json and not any(validator.iter_errors(parsed))
    matches = field_matches(row["cikti"], parsed if valid_json else None)
    records.append(
        {
            "valid_json": valid_json,
            "schema_ok": schema_ok,
            "field_match": matches,
            "gen": gen,
            "parsed": parsed,
        }
    )
    print(f"{i + 1}/{N_EVAL} json={valid_json} sema={schema_ok}")


def rate(vals):
    return sum(1 for v in vals if v) / len(vals) if vals else 0.0


n = len(records)
summary = OrderedDict(
    [
        ("n", n),
        ("geçerli JSON", rate([r["valid_json"] for r in records])),
        ("şema uyum", rate([r["schema_ok"] for r in records])),
    ]
)
for name in CORE_FIELDS:
    summary[name] = rate([r["field_match"][name] for r in records])

headers = list(summary.keys())
values = [str(summary["n"])] + [f"{100 * v:5.1f}%" for v in list(summary.values())[1:]]
widths = [max(len(h), len(v)) for h, v in zip(headers, values)]
print()
print("  ".join(h.ljust(w) for h, w in zip(headers, widths)))
print("  ".join("-" * w for w in widths))
print("  ".join(v.ljust(w) for v, w in zip(values, widths)))

print("\n=== en sık şema hataları ===")
sayac = {}
for r in records:
    if not r["parsed"]:
        continue
    for h in validator.iter_errors(r["parsed"]):
        anahtar = f"{'/'.join(str(p) for p in h.path) or '(kok)'}: {h.validator}"
        sayac[anahtar] = sayac.get(anahtar, 0) + 1
for k, v in sorted(sayac.items(), key=lambda x: -x[1])[:10]:
    print(f"  {v:3d}  {k}")



In [ ]:
# Hücre 6 — yayınlama
# Adapter ve merge edilmiş model ayrı HF repolarına gider.
# Inference Endpoint dağıtımı merged sürümü kullanır.
#
# Not: Bu hücre eğitimden hemen sonra çalıştırılmalıdır. Colab oturumu
# düşerse bellekteki model kaybolur; HF'ye yüklenmiş ağırlık kalıcıdır.

import gc
from pathlib import Path

import torch
from huggingface_hub import HfApi
from peft import PeftModel
from transformers import AutoModelForCausalLM

HF_USER = "fatmaoz"
ADAPTER_REPO = f"{HF_USER}/kontrata-qwen-lora-v1"
MERGED_REPO = f"{HF_USER}/kontrata-qwen-merged-v1"

ADAPTER_CARD = """---
base_model: Qwen/Qwen2.5-1.5B-Instruct
library_name: peft
license: apache-2.0
tags:
- lora
- qwen2.5
- text-generation
---

# kontrata-qwen-lora-v1

Qwen2.5-1.5B-Instruct üzerine LoRA (r=16, alpha=32) ile eğitilmiş adapter.
Inference Endpoint dağıtımı için birleşik repoyu kullanın: `fatmaoz/kontrata-qwen-merged-v1`.

## Amaç

Otel-acente kontenjan sözleşmesi düz yazısından `kontrat.json` şemasına uygun JSON üretmek
(Okuyucu agent). Sözleşme verisi tesisten çıkmaz; bu ağırlık yalnızca çıkarım motorudur.

## Eğitim

- Taban model: `Qwen/Qwen2.5-1.5B-Instruct`, 4-bit nf4 (double quant)
- Yöntem: PEFT/LoRA, r=16, alpha=32, dropout=0.05
- Veri: 320 sentetik eğitim + 80 doğrulama örneği (`ml/generate.py --seed 42`)
- 4 epoch, val kaybı 0.157, ortalama token doğruluğu %94.5

Şema MEB MEGEP "Paket Tur Üretimi" modülündeki örnek kontenjan sözleşmesi ve
Turizm İşletmeleri Yönetmeliği'nden türetilmiştir. Gerçek otel sözleşmesi kullanılmadı.
MEGEP Argos örneği eğitim kümesine alınmamıştır; yalnızca test setindedir.

## Bilinen sınırlamalar

- Ham çıktı JSON olarak geçerlidir (%100) ancak şemaya doğrudan uymaz: fazladan alan,
  tip hataları ve eksik zorunlu alanlar üretir. Üretimde onarım ve doğrulama katmanıyla
  birlikte kullanılmalıdır (`backend/internal/extract`).
- Sentetik veriyle eğitildi; gerçek operatör sözleşmelerinde performans ölçülmedi.
- Yalnızca Türkçe ve İngilizce metinlerde denendi.
- `opsiyonel` alanlar (iptal, çocuk politikası, ödeme) kapsam dışıdır.
- Uzun ekler 1024 tokende kesilir.
"""

MERGED_CARD = """---
base_model: Qwen/Qwen2.5-1.5B-Instruct
license: apache-2.0
tags:
- qwen2.5
- text-generation
---

# kontrata-qwen-merged-v1

`kontrata-qwen-lora-v1` adapterinin taban modele birleştirilmiş hali.
HuggingFace Inference Endpoint dağıtımı için bu repo kullanılır.

## Taban model

`Qwen/Qwen2.5-1.5B-Instruct`

## Eğitim verisi

Sentetik 320 örnek. MEGEP Argos örneği eğitimde yoktur.

## Bilinen sınırlamalar

Adapter kartındaki sınırlamalar geçerlidir. Birleşik ağırlık 4-bit eğitimden gelir;
kalite tam kesinlik LoRA ile aynı kabul edilmemelidir. Üretimde maskeleme katmanı
modelden önce çalışır.
"""

api = HfApi()

print(f"push adapter -> {ADAPTER_REPO}")
Path(ADAPTER_DIR).mkdir(parents=True, exist_ok=True)
(Path(ADAPTER_DIR) / "README.md").write_text(ADAPTER_CARD, encoding="utf-8")
api.create_repo(ADAPTER_REPO, exist_ok=True, private=True)
api.upload_folder(folder_path=ADAPTER_DIR, repo_id=ADAPTER_REPO)

for isim in ("model", "trainer"):
    if isim in globals():
        del globals()[isim]
gc.collect()
torch.cuda.empty_cache()

print("adapter birleştiriliyor (CPU, tam kesinlik)")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.bfloat16,
    device_map="cpu",
    trust_remote_code=True,
)
peft_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
merged = peft_model.merge_and_unload()

Path(MERGED_DIR).mkdir(parents=True, exist_ok=True)
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
(Path(MERGED_DIR) / "README.md").write_text(MERGED_CARD, encoding="utf-8")

print(f"push merged -> {MERGED_REPO}")
api.create_repo(MERGED_REPO, exist_ok=True, private=True)
api.upload_folder(folder_path=MERGED_DIR, repo_id=MERGED_REPO)
print("tamam")
